# ⚡ Workshop Databricks — Time de Negócio | Setor de Energia
### Guia Passo a Passo

Bem-vindo(a)! Neste workshop você vai aprender a usar ferramentas da Databricks para **explorar, transformar e visualizar dados de energia**.

**Agenda do dia:**

| # | Tópico | O que você vai fazer |
|---|---|---|
| 1 | Upload de arquivos | Fazer upload de CSVs e criar tabelas |
| 2 | Unity Catalog | Explorar catálogo, schema e volume |
| 3 | SQL Básico + AI Functions | Escrever queries e usar IA dentro do SQL |
| 4 | Lakeflow Designer | Construir um pipeline visual de dados |
| 5 | Genie Spaces | Fazer perguntas em linguagem natural |
| 6 | Databricks One | Visão geral da plataforma |
| 7 | AI/BI | Criar dashboards com linguagem natural |


## Descrição das tabelas
### 📋 Descrição das Colunas — Tabela CLIENTES

 Tabela      | Coluna                | Descrição                                                                 |
-------------|----------------------|---------------------------------------------------------------------------|
 clientes    | cliente_id           | Identificador único do cliente                                            |
 clientes    | nome                 | Nome do cliente                                                           |
 clientes    | tipo_cliente         | Tipo de cliente (Residencial, Comercial, Industrial)                      |
 clientes    | cidade               | Cidade do cliente                                                         |
 clientes    | estado               | Estado do cliente                                                         |
 clientes    | tensao               | Nível de tensão da ligação                                                |
 clientes    | data_cadastro        | Data de cadastro do cliente                                               |
 clientes    | ativo                | Status de atividade do cliente                                            |

### 📋 Descrição das Colunas — Tabela CONSUMO

 Tabela      | Coluna                | Descrição                                                                 |
-------------|----------------------|---------------------------------------------------------------------------|
 consumo     | consumo_id           | Identificador único do registro de consumo                                |
 consumo     | cliente_id           | Identificador do cliente                                                  |
 consumo     | data_leitura         | Data da leitura                                                           |
 consumo     | mes                  | Mês da leitura                                                            |
 consumo     | ano                  | Ano da leitura                                                            |
 consumo     | kwh_consumido        | Quantidade de energia consumida (kWh)                                     |
 consumo     | demanda_kw           | Demanda de energia (kW)                                                   |
 consumo     | valor_rs             | Valor da fatura em reais                                                  |
 consumo     | status_fatura        | Status da fatura (Paga, Pendente, Vencida)                                |

### 📋 Descrição das Colunas — Tabela INSTALACOES

 Tabela      | Coluna                | Descrição                                                                 |
-------------|----------------------|---------------------------------------------------------------------------|
 instalacoes | instalacao_id         | Identificador único da instalação                                         |
 instalacoes | cliente_id            | Identificador do cliente                                                  |
 instalacoes | tipo_equipamento      | Tipo de equipamento instalado                                             |
 instalacoes | fabricante            | Fabricante do equipamento                                                 |
 instalacoes | capacidade_kw         | Capacidade do equipamento (kW)                                            |
 instalacoes | data_instalacao       | Data de instalação                                                        |
 instalacoes | ultima_manutencao     | Data da última manutenção                                                 |
 instalacoes | status                | Status do equipamento                                                     |
 instalacoes | vida_util_anos        | Vida útil estimada em anos                                                |

---
## 📁 Módulo 1 — Upload de Arquivos e Criação de Tabelas

### O que é?
O Databricks permite fazer upload de arquivos (XLS, CSV, JSON) diretamente pela interface e convertê-los automaticamente em tabelas Delta no Unity Catalog — sem escrever uma linha de código.

### Passo a passo:

1. No menu lateral esquerdo, clique em **"+ New"** → **"Add or upload data"**
2. Arraste o arquivo CSV ou clique para selecionar o arquivo do seu computador
3. O Databricks detecta automaticamente o schema (tipos das colunas)
4. Revise os tipos de dados e corrija se necessário
5. Escolha o **Catálogo**, **Schema** e dê um **nome** para a tabela
6. Clique em **"Create table"**

### Arquivos para fazer upload (gerados no Notebook 1):
- `clientes.csv` → Tabela: `clientes`
- `consumo.csv` → Tabela: `consumo`
- `instalacoes.csv` → Tabela: `instalacoes`

> 💡 **Dica:** Para arquivos Excel (`.xlsx`), o processo é o mesmo. O Databricks converte cada aba em uma tabela separada.

---
## 🗂️ Módulo 2 — Unity Catalog: Catálogo, Schema e Volume

### O que é o Unity Catalog?
O Unity Catalog é o sistema de governança de dados da Databricks. Pense nele como um **gerenciador de arquivos inteligente** para dados, com três níveis hierárquicos:

```
Catálogo (Catalog)
└── Schema (Database)
    ├── Tabela (Table)   ← dados estruturados (linhas e colunas)
    ├── Volume           ← arquivos brutos (CSV, JSON, imagens, PDFs)
    └── View             ← consulta salva como "tabela virtual"
```

| Conceito | Analogia | Exemplo |
|---|---|---|
| **Catálogo** | Pasta principal | `main` |
| **Schema** | Subpasta por projeto | `workshop_energia` |
| **Tabela** | Planilha Excel | `clientes`, `consumo` |
| **Volume** | Área de arquivos brutos | `/Volumes/main/workshop_energia/arquivos` |

### Como explorar no Unity Catalog:
1. No menu lateral, clique em **"Catalog"** (ícone de livro)
2. Expanda seu catálogo → schema → veja as tabelas criadas
3. Clique em uma tabela para ver **schema**, **sample data** e **lineage**
4. Na aba **"Volumes"**, veja os arquivos CSVs que foram carregados

### Vamos configurar algumas váriaveis só para facilitar o passo a passo do lab

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("nome_catalogo","","Catálogo")
dbutils.widgets.text("nome_schema","","Schema")
dbutils.widgets.text("seu_prefixo","","Seu Prefixo")
dbutils.widgets.text("caminho_volume", "", "Caminho do Volume")

In [0]:
nome_catalogo  = dbutils.widgets.get("nome_catalogo")
nome_schema    = dbutils.widgets.get("nome_schema")
seu_prefixo    = dbutils.widgets.get("seu_prefixo")
caminho_volume = dbutils.widgets.get("caminho_volume")

print(f"Catálogo : {nome_catalogo}")
print(f"Schema   : {nome_schema}")
print(f"Prefixo  : {seu_prefixo}")
print(f"Volume   : {caminho_volume}")

---
## 💻 Módulo 3 — SQL Básico, AI Functions, Notebook e SQL Editor

### 3.1 — SQL Editor
O SQL Editor é a ferramenta para escrever e executar queries SQL interativamente.
**Como acessar:** Menu lateral → **"SQL Editor"**

---

### 📝 Queries de Exemplo — Execute no SQL Editor

#### Query 1 — Exploração básica: Ver os primeiros clientes

In [0]:
%sql
    
SELECT *
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_clientes`
LIMIT 10;

#### Query 2 — Contagem por tipo de cliente

In [0]:
%sql
    
SELECT
  tipo_cliente,
  COUNT(*)          AS total_clientes,
  COUNT(CASE WHEN ativo = true THEN 1 END) AS clientes_ativos
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_clientes`
GROUP BY tipo_cliente
ORDER BY total_clientes DESC;

#### Query 3 — Consumo médio mensal por tipo de cliente

In [0]:
%sql
    
SELECT
  c.tipo_cliente,
  ROUND(AVG(con.kwh_consumido), 2) AS media_kwh,
  ROUND(AVG(con.valor_rs), 2)      AS media_valor_rs,
  ROUND(SUM(con.valor_rs), 2)      AS total_faturado_rs
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_consumo` con
JOIN `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_clientes` c
  ON con.cliente_id = c.cliente_id
GROUP BY c.tipo_cliente
ORDER BY total_faturado_rs DESC;

#### Query 4 — Top 10 clientes por gasto total

In [0]:
%sql
    
SELECT
  c.nome,
  c.tipo_cliente,
  c.cidade,
  c.estado,
  ROUND(SUM(con.valor_rs), 2)    AS total_gasto_rs,
  ROUND(SUM(con.kwh_consumido), 2) AS total_kwh
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_consumo` con
JOIN `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_clientes` c
  ON con.cliente_id = c.cliente_id
GROUP BY c.nome, c.tipo_cliente, c.cidade, c.estado
ORDER BY total_gasto_rs DESC
LIMIT 10;

#### Query 5 — Regra de negócio: Classificação de clientes por tier (Ouro / Prata / Bronze)
> **Regra:** Clientes com gasto médio mensal > R$5.000 = Ouro | > R$1.000 = Prata | demais = Bronze

In [0]:
%sql
    
WITH gasto_mensal AS (
  SELECT
    c.cliente_id,
    c.nome,
    c.tipo_cliente,
    c.cidade,
    ROUND(AVG(con.valor_rs), 2) AS media_mensal_rs
  FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_consumo` con
  JOIN `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_clientes` c
    ON con.cliente_id = c.cliente_id
  GROUP BY c.cliente_id, c.nome, c.tipo_cliente, c.cidade
)
SELECT
  *,
  CASE
    WHEN media_mensal_rs > 5000 THEN 'Ouro'
    WHEN media_mensal_rs > 1000 THEN 'Prata'
    ELSE 'Bronze'
  END AS tier_cliente
FROM gasto_mensal
ORDER BY media_mensal_rs DESC;

#### Query 6 — Faturas vencidas: inadimplência por estado

In [0]:
%sql
    
SELECT
  c.estado,
  COUNT(*)                                              AS total_faturas,
  COUNT(CASE WHEN con.status_fatura = 'Vencida' THEN 1 END) AS faturas_vencidas,
  ROUND(
    COUNT(CASE WHEN con.status_fatura = 'Vencida' THEN 1 END) * 100.0 / COUNT(*), 1
  )                                                     AS pct_inadimplencia,
  ROUND(SUM(CASE WHEN con.status_fatura = 'Vencida' THEN con.valor_rs ELSE 0 END), 2) AS valor_em_atraso_rs
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_consumo` con
JOIN `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_clientes` c
  ON con.cliente_id = c.cliente_id
GROUP BY c.estado
ORDER BY valor_em_atraso_rs DESC;

#### Query 7 — Equipamentos que precisam de manutenção (última manutenção há mais de 1 ano)

In [0]:
%sql
    
SELECT
  i.instalacao_id,
  c.nome                           AS cliente,
  c.tipo_cliente,
  i.tipo_equipamento,
  i.fabricante,
  i.ultima_manutencao,
  DATEDIFF(CURRENT_DATE(), i.ultima_manutencao) AS dias_sem_manutencao,
  i.status
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_instalacoes` i
JOIN `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_clientes` c
  ON i.cliente_id = c.cliente_id
WHERE DATEDIFF(CURRENT_DATE(), i.ultima_manutencao) > 365
  AND i.status = 'Ativo'
ORDER BY dias_sem_manutencao DESC;

---
### 3.2 — AI Functions no SQL
As AI Functions permitem usar **IA generativa diretamente dentro de queries SQL**.
A função mais usada é `ai_gen()`, que gera texto com base em um prompt.

#### Query 8 — `ai_gen`: Gerar análise automática de cliente em linguagem natural

In [0]:
%sql
    
-- Exemplo: gerar resumo automático do perfil de cada cliente
SELECT
  c.nome,
  c.tipo_cliente,
  c.cidade,
  ROUND(SUM(con.valor_rs), 2)      AS total_gasto_rs,
  ROUND(AVG(con.kwh_consumido), 2) AS media_kwh,
  ai_gen(
    CONCAT(
      'Você é um analista de uma distribuidora de energia. Escreva um resumo executivo em 2 frases sobre o cliente a seguir. ',
      'Nome: ', c.nome, '. ',
      'Tipo: ', c.tipo_cliente, '. ',
      'Cidade: ', c.cidade, '. ',
      'Gasto total: R$ ', ROUND(SUM(con.valor_rs), 2), '. ',
      'Consumo médio mensal: ', ROUND(AVG(con.kwh_consumido), 2), ' kWh. ',
      'Destaque os pontos mais relevantes para a equipe comercial.'
    )
  ) AS resumo_ia
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_consumo` con
JOIN `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_clientes` c
  ON con.cliente_id = c.cliente_id
GROUP BY c.nome, c.tipo_cliente, c.cidade
ORDER BY total_gasto_rs DESC
LIMIT 5;

#### Query 9 — `ai_classify`: Classificar status de equipamento com IA

In [0]:
%sql
    
-- ai_classify categoriza texto em classes predefinidas
SELECT
  tipo_equipamento,
  fabricante,
  status,
  DATEDIFF(CURRENT_DATE(), ultima_manutencao) AS dias_sem_manutencao,
  ai_classify(
    CONCAT('Equipamento: ', tipo_equipamento, '. Status: ', status,
           '. Dias sem manutenção: ', DATEDIFF(CURRENT_DATE(), ultima_manutencao)),
    ARRAY('Urgente', 'Atenção', 'Normal')
  ) AS prioridade_manutencao
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_instalacoes`
LIMIT 20;

### 3.3 Criando tabelas em SQL com base no path de um arquivo do volume (opcional)

#### Query 10 — Usando SQL para criar tabela com base num caminho de arquivo CSV do volume

In [0]:
%sql
CREATE TABLE nome_catalogo.nome_schema.nome_sua_tabela
USING CSV
OPTIONS (header = 'true', inferSchema = 'true')
LOCATION '/Volumes/seu_volume/seu_arquivo.csv'


#### Query 11 — Usando SQL para criar tabela com base num caminho de arquivo XLS do volume

In [0]:
%sql
CREATE TABLE nome_catalogo.nome_schema.nome_sua_tabela
USING EXCEL
OPTIONS (header = 'true', inferSchema = 'true')
LOCATION '/Volumes/seu_volume/seu_arquivo.xlsx'

---
## 🔄 Módulo 4 — Lakeflow Designer: Pipeline Visual de Dados

### O que é o Lakeflow Designer?
O Lakeflow Designer é uma ferramenta **visual (drag-and-drop)** para construir pipelines de transformação de dados — sem escrever código. Ideal para times de negócio que precisam criar fluxos de dados com lógica de negócio.

**Como acessar:** Menu lateral → **"+New"** → **"Visual data prep"** 

---
### 🤖 Usando o Genie Code para Criar Nós no Lakeflow Designer

O **Genie Code** permite descrever em linguagem natural o que você quer fazer, e ele gera automaticamente a configuração do nó no Lakeflow Designer.

**Como usar:**
1. No Lakeflow Designer, clique no ícone **"AI Assistant"** (varinha mágica ✨)
2. Digite as instrução em português ou inglês
3. Revise o nó gerado e confirme

### Instruções

- Faça um join entre a tabela seuprefixo_consumo e a tabela seuprefixo_clientes usando o campo cliente_id
- Filtre apenas os registros onde status_fatura é igual a Paga
- Some o kwh_consumido e o valor_rs agrupando por tipo_cliente e estado
- Crie uma coluna chamada tier_cliente: se valor_rs médio for maior que 5000 retorna Ouro,
 se maior que 1000 retorna Prata, caso contrário retorna Bronze
- Salve o resultado na tabela nome_catalogo.nome_schema.seuprefixo_consumo_consolidado 

---

### 4.1 — Adicionar Data Sources (fontes de dados)

**Passo a passo:**
1. No canvas do Lakeflow, clique em **"+ Add source"**
2. Selecione **"Unity Catalog table"**
3. Busque e selecione a tabela `clientes`
4. Repita para adicionar `consumo` e `instalacoes`

> Você verá 3 nós de entrada no canvas, um para cada tabela.

---

### 4.2 — Join de Tabelas

**Passo a passo:**
1. Arraste uma conexão da tabela `consumo` para um novo nó **"Join"**
2. Conecte também a tabela `clientes` ao mesmo nó Join
3. Configure o Join:
   - **Join type:** Inner Join
   - **Left key:** `consumo.cliente_id`
   - **Right key:** `clientes.cliente_id`
4. Selecione as colunas que deseja manter no resultado

---

### 4.3 — Filtrar Dados

**Passo a passo:**
1. Após o Join, adicione um nó **"Filter"**
2. Configure a condição de filtro. Exemplos:
   - Apenas faturas pagas: `status_fatura = 'Paga'`

---

### 4.4 — Agregação (Soma, Média, Contagem)

**Passo a passo:**
1. Adicione um nó **"Aggregate"**
2. Configure:
   - **Group by:** `tipo_cliente`, `estado` (ou `cidade`, `ano`, `mes`)
   - **Aggregations:**
     - `SUM(kwh_consumido)` → nome: `total_kwh`
     - `SUM(valor_rs)` → nome: `total_faturado_rs`
     - `COUNT(consumo_id)` → nome: `qtd_faturas`
     - `AVG(valor_rs)` → nome: `media_fatura_rs`

---

### 4.5 — Transformação com Regra de Negócio

**Passo a passo:**
1. Adicione um nó **"Derived column"** (coluna derivada)
2. Crie uma nova coluna chamada `tier_cliente` com a fórmula:
   ```
   CASE
     WHEN media_fatura_rs > 5000 THEN 'Ouro'
     WHEN media_fatura_rs > 1000 THEN 'Prata'
     ELSE 'Bronze'
   END
   ```
3. Outra coluna útil: `status_manutencao`:
   ```
   CASE
     WHEN dias_sem_manutencao > 365 THEN 'Atrasada'
     WHEN dias_sem_manutencao > 180 THEN 'Próxima'
     ELSE 'Em dia'
   END
   ```

---

### 4.6 — Escrever Resultado em Tabela de Saída

**Passo a passo:**
1. Adicione um nó **"Write to table"** ao final do pipeline
2. Configure:
   - **Catalog:** `seu_catalogo`
   - **Schema:** `seu_schema`
   - **Table name:** `seuprefixo_consumo_por_cliente_tier` (ou o nome que preferir)
   - **Write mode:** `Overwrite` (substitui a tabela a cada execução)
3. Clique em **"Save"**

---

### 4.7 — Agendar Execução do Pipeline (Job)

**Passo a passo:**
1. No Lakeflow Designer, clique em **"Schedule"** (canto superior direito)
2. Escolha a frequência:
   - **Manual:** executar sob demanda (botão "Run")
   - **Scheduled:** definir horário (ex: todo dia às 8h, toda segunda-feira)
   - **File arrival:** executar quando um novo arquivo chegar no Volume
3. Configure alertas de email em caso de falha
4. Clique em **"Save schedule"**

> 💡 **Dica:** Você pode monitorar as execuções passadas em **"Workflow Runs"** → veja duração, status e logs de cada etapa.

---
## 🧞 Módulo 5 — Genie Spaces: Dados em Linguagem Natural

### O que é o Genie Space?
O Genie Space é um **assistente de BI conversacional** da Databricks. Você faz perguntas em português (ou inglês) sobre seus dados e ele responde com tabelas, gráficos e explicações — sem precisar escrever SQL.

**Como acessar:** Menu lateral → **"Genie"** → selecione ou crie um Space configurado com suas tabelas.

### Selecionando as tabelas
Seleciona as 4 tabelas com seu_prefixo criadas ao longo do workshop

---

### 5.1 — Perguntas Comuns (modo padrão)

Copie e cole estas perguntas diretamente no Genie Space:

**Exploração geral:**
```
Quantos clientes temos no total? Quebre por tipo de cliente.
```
```
Qual estado tem mais clientes industriais?
```
```
Qual foi o mês com maior consumo de energia em 2025?
```
```
Mostre o total faturado por estado em ordem decrescente.
```
```
Quais são os 10 clientes com maior consumo de energia?
```

**Inadimplência:**
```
Quantas faturas estão vencidas? Qual é o valor total em atraso?
```
```
Qual tipo de cliente tem maior taxa de inadimplência?
```
```
Liste os clientes com faturas vencidas e o valor total de cada um.
```

**Equipamentos:**
```
Quais equipamentos estão com status Em Manutenção?
```
```
Quantos equipamentos não receberam manutenção nos últimos 12 meses?
```
```
Qual fabricante tem mais equipamentos instalados?
```

---

### 5.2 — Perguntas Hipotéticas com o Modo Agent (Deep Research)

O modo **Agent** do Genie permite fazer **análises hipotéticas e exploratórias** mais profundas. Ative-o clicando em "Agent mode" dentro do Genie Space.

**Exemplos de perguntas hipotéticas:**

```
- Se aumentarmos a tarifa de Baixa Tensão em 10%, qual seria o impacto no faturamento total? Quais clientes residenciais seriam mais afetados?
```

```
- Se todos os clientes com faturas vencidas quitassem suas dívidas hoje, qual seria o aumento percentual na receita do mês atual?
```

```
- Baseado no histórico de consumo, quais clientes industriais têm maior risco de aumentar o consumo acima da demanda contratada no próximo trimestre?
```

```
- Se priorizarmos manutenção preventiva nos equipamentos Siemens e ABB com mais de 365 dias sem revisão, quantos clientes seriam impactados e qual seria o custo estimado considerando R$1.500 por visita técnica?
```

```
- Analise a sazonalidade do consumo dos últimos 2 anos. Quando devo aumentar a capacidade de atendimento e em quais regiões?
```

```
- Se migrarmos todos os clientes Comerciais de Média Tensão para Alta Tensão, qual seria a economia gerada na tarifa e qual o impacto na receita da distribuidora?
```

```
- Crie um ranking de risco de churn dos clientes com base em: consumo declinante, histórico de inadimplência e equipamentos com status Desativado.
```

---
## 🌐 Módulo 6 — Databricks One: Visão Geral da Plataforma

### O que é?
O **Databricks One** é a experiência unificada da plataforma Databricks que integra em um único ambiente:

| Ferramenta | Função |
|---|---|
| **Unity Catalog** | Governança e descoberta de dados |
| **SQL Editor** | Análise ad-hoc em SQL |
| **Notebooks** | Desenvolvimento em Python/SQL/R |
| **Lakeflow Designer** | Pipelines visuais de dados |
| **Genie Spaces** | BI conversacional com IA |
| **AI/BI Dashboards** | Dashboards interativos |
| **Workflows** | Orquestração e agendamento de jobs |
| **Marketplace** | Dados e modelos prontos para uso |

### Principais diferenciais:
- **Segurança centralizada:** controle de acesso por catálogo, schema e tabela
- **Colaboração:** notebooks e dashboards podem ser compartilhados com o time
- **Escalabilidade:** processa desde milhares até bilhões de registros
- **IA integrada:** AI Functions, Genie, e modelos de ML na mesma plataforma

> 💡 **Para times de negócio:** Você não precisa conhecer toda a plataforma. Comece com SQL Editor e Genie — são suficientes para 80% das análises do dia a dia.

---
## 📊 Módulo 7 — AI/BI: Dashboards com Linguagem Natural

### O que é o AI/BI?
O **AI/BI** é o sistema de dashboards da Databricks que permite:
- Criar visualizações descrevendo o que você quer em linguagem natural
- Explorar dashboards fazendo perguntas diretamente para a IA
- Criar gráficos sem precisar configurar manualmente eixos, filtros e agrupamentos

**Como acessar:** Menu lateral → **"Dashboards"** → **"New dashboard"**

---

### 7.1 — Criar Gráficos com Linguagem Natural

**Passo a passo:**
1. Clique em **"+ Add"** → **"Visualization"**
2. No campo de prompt, descreva o gráfico que deseja criar. Exemplos:

```
Crie um gráfico de barras mostrando o total faturado por tipo de cliente
```
```
Mostre a evolução mensal do consumo de energia (kWh) por ano como um gráfico de linha
```

```
Mostre um gráfico de pizza com a distribuição dos status de fatura
(Paga, Pendente, Vencida)
```
```
Crie um gráfico de barras empilhadas com o consumo total por mês e por tipo de cliente
```
```
Mostre um scatter plot com kwh_consumido no eixo X e valor_rs no eixo Y,
colorido por tipo_cliente
```
```
Crie um ranking dos 10 estados com maior valor de faturas vencidas
```
```
Mostre a distribuição dos equipamentos por fabricante em um gráfico de barras horizontais
```

**Dica:** Adicione um **filtro de data** para tornar o dashboard interativo:
- Clique em **"+ Add filter"** → selecione o campo `data_leitura` ou `ano`

---

### Utilizando o AI/BI + Genie Code para criar vários gráficos

É possível utilizar o Genie Code para criar vários gráficos, edições, e abas ao mesmo tempo. Seja claro no seu prompt sobre que tipo de métricas, estatísticas você quer reproduzir.

```
Crie um dashboard completo chamado "Painel de Gestão Energética" que conte a história do consumo de energia dos nossos clientes, organizado em 4 páginas:

Página 1 - "Visão Geral": Quero uma visão executiva do negócio. Mostre os principais KPIs: total de clientes ativos, total de kWh consumidos e o faturamento total em R$. Depois, mostre como os clientes estão distribuídos por tipo (Industrial, Comercial, Residencial) e qual é o faturamento por estado brasileiro.

Página 2 - "Análise de Consumo": Preciso entender os padrões de consumo ao longo do tempo. Mostre como o consumo de energia evoluiu mês a mês ao longo dos anos (2022, 2023, 2024). Quero entender também como cada tipo de cliente contribui para o consumo mensal total. Além disso, mostre a evolução do valor faturado por ano e quem são os 10 maiores consumidores.

Página 3 - "Infraestrutura": Quero analisar nosso parque de equipamentos. Quantas instalações temos por tipo de equipamento? Como estão distribuídas por status (ativo, inativo, manutenção)? Qual a capacidade instalada por fabricante? E qual a média de vida útil dos nossos equipamentos?

Página 4 - "Inadimplência": Preciso monitorar a saúde financeira. Como estão as faturas por status (paga, pendente, atrasada)? Existe diferença de inadimplência entre os tipos de cliente? Qual o valor total em risco com faturas pendentes? E como a inadimplência está evoluindo ao longo dos meses?

Adicione filtros globais para tipo de cliente e estado que afetem todas as páginas. Escolha os tipos de gráficos mais adequados para cada análise. Use um tema profissional com cores harmoniosas.

```
---

### 7.2 — Explorar os Gráficos com Perguntas para o AI/BI

Após criar o dashboard, clique no ícone de **chat com IA** (canto inferior direito) e explore:

**Perguntas de análise:**
```
Por que o consumo de outubro a dezembro é sempre maior?
```
```
Quais estados apresentam crescimento de consumo acima da média nacional?
```
```
Existe correlação entre o tipo de equipamento instalado e o consumo mensal?
```
```
O gráfico mostra sazonalidade? Em quais meses o consumo é maior?
```

**Perguntas de negócio:**
```
Qual tipo de cliente é mais rentável para a empresa?
```
```
Quais estados merecem maior atenção comercial pelo volume de clientes
com alto potencial de consumo?
```
```
Baseado no gráfico de inadimplência, em quais regiões devemos intensificar
a cobrança?
```
```
Estou vendo um pico de consumo em janeiro. O que pode explicar isso?
```

**Perguntas de drill-down:**
```
Clique no estado SP no gráfico — quais cidades contribuem mais para o consumo?
```
```
Filtre apenas clientes Industriais. Como muda o ranking de estados?
```
```
Compare o faturamento de 2024 vs 2025. Houve crescimento?
```

---

### 7.3 — Compartilhar o Dashboard

1. Clique em **"Share"** (canto superior direito)
2. Adicione membros do time ou gere um link público
3. Defina permissões: **Can view** (apenas visualizar) ou **Can edit** (editar)
4. O dashboard atualiza automaticamente conforme novas tabelas são gravadas pelo Lakeflow

---
## 🎯 Resumo do Workshop

Parabéns! Você completou o workshop. Aqui está o que você aprendeu:

| Módulo | Habilidade adquirida |
|---|---|
| 1 | Fazer upload de arquivos e criar tabelas no Databricks |
| 2 | Navegar no Unity Catalog e entender a hierarquia de dados |
| 3 | Escrever queries SQL com lógica de negócio e usar AI Functions |
| 4 | Construir pipelines de dados visuais no Lakeflow Designer |
| 5 | Fazer perguntas sobre dados em linguagem natural no Genie |
| 6 | Conhecer a plataforma Databricks One como um todo |
| 7 | Criar dashboards interativos com IA no AI/BI |

### Próximos passos sugeridos:
1. **Traga seus próprios dados** — faça upload de um XLS do seu time e repita o exercício
2. **Configure um Genie Space** com as tabelas do seu projeto real
3. **Automatize** o pipeline do Lakeflow para rodar diariamente
4. **Compartilhe** o dashboard com stakeholders da empresa

> 📬 **Dúvidas?** Fale com a equipe Databricks ou acesse a documentação em [docs.databricks.com](https://docs.databricks.com)